# 需要上传预处理过的数据。data_train.json 和 data_validation.json

In [ ]:
import datetime
import os
import json
import torch
import pandas as pd
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup, RobertaModel, RobertaConfig, AutoTokenizer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
import torch.nn as nn
from tqdm import tqdm
import numpy as np
from collections import defaultdict
import logging
from typing import Callable, List # Added for type hints in utility functions

# ==================== Colab 路径配置 ====================
# 数据目录 (上传数据文件到此目录)
DATA_DIR = "/content/"
# 使用 Hugging Face 在线模型
MODEL_PATH = "huggingface/CodeBERTa-small-v1"
# 模型保存目录
SAVED_MODEL_DIR = "/content/saved_models/"

# ==================== 训练超参数 ====================
NUM_LABELS = 9
EPOCHS = 10
BATCH_SIZE = 8  # Reduced from 32 to 8 to mitigate CUDA out of memory errors
LEARNING_RATE = 5e-5
MAX_LENGTH = 512 # Max sequence length for tokenizer, added from inference script

# ==================== 标签映射 ====================
LABEL_MAP = {
    "select_a": 0,
    "select_b": 1,
    "select_o": 2,
    "concat_ab": 3,
    "concat_ba": 4,
    "select_a_del_o": 5,
    "select_b_del_o": 6,
    "concat_ab_del_o": 7,
    "concat_ba_del_o": 8,
}
REVERSE_LABEL_MAP = {v: k for k, v in LABEL_MAP.items()}


# ==================== 模型定义 (内嵌 model.py) ====================
class MergeBERTModel(nn.Module):
    """MergeBERT model for merge conflict resolution."""
    def __init__(self, encoder_path, num_labels=9):
        super(MergeBERTModel, self).__init__()
        self.num_labels = num_labels

        # The paper describes 4 parallel encoders. We'll instantiate one and use it four times.
        # This enforces weight sharing across the encoders, which is a common practice.
        config = RobertaConfig.from_pretrained(encoder_path)
        # Load the model in full-precision (float32) by default. AMP will handle mixed precision.
        self.encoder = RobertaModel.from_pretrained(encoder_path, config=config)

        # Aggregation weights (learnable) - default to float32
        self.aggregation_weights = nn.Parameter(torch.ones(4))

        # Classification head - default to float32
        self.classifier = nn.Linear(config.hidden_size, num_labels)

    def forward(self, a_o_ids, o_a_ids, b_o_ids, o_b_ids, labels=None):
        """
        Forward pass of the model.
        For the MVP, we are not using the edit type embeddings yet.
        `*_ids` are tensors of shape (batch_size, seq_length).
        """
        # Pass each input through the shared encoder
        output_ao = self.encoder(input_ids=a_o_ids).pooler_output
        output_oa = self.encoder(input_ids=o_a_ids).pooler_output
        output_bo = self.encoder(input_ids=b_o_ids).pooler_output
        output_ob = self.encoder(input_ids=o_b_ids).pooler_output

        # Weighted aggregation of the [CLS] token representations (pooler_output)
        agg_weights = nn.functional.softmax(self.aggregation_weights, dim=0)

        aggregated_output = (
            agg_weights[0] * output_ao +
            agg_weights[1] * output_oa +
            agg_weights[2] * output_bo +
            agg_weights[3] * output_ob
        )

        # Classification
        logits = self.classifier(aggregated_output)

        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))

        return {
            "loss": loss,
            "logits": logits
        }


# ==================== Utility functions for alignment (copied from inferencecolab.py) ====================
edit_seq_tokens = {
    "eql": "=", "add": "+", "del": "-", "rep": "↔", "padding": "[PAD]"
}

class OffsetRange:
    def __init__(self, start: int, end: int):
        self.start, self.end = start, end
    def __len__(self):
        return self.end - self.start

class SequenceDiff:
    def __init__(self, seq1Range: OffsetRange, seq2Range: OffsetRange):
        self.seq1Range, self.seq2Range = seq1Range, seq2Range

def compute(sequence1: List[str], sequence2: List[str], equalityScore: Callable[[str, str], float] = lambda x, y: 1 if x == y else 0) -> List[SequenceDiff]:
    if not sequence1 or not sequence2: return [SequenceDiff(OffsetRange(0, len(sequence1)), OffsetRange(0, len(sequence2)))]
    lcsLengths, directions, lengths = np.zeros((len(sequence1), len(sequence2))), np.zeros((len(sequence1), len(sequence2))), np.zeros((len(sequence1), len(sequence2)))
    for s1 in range(len(sequence1)): # type: ignore
        for s2 in range(len(sequence2)): # type: ignore
            h_len, v_len = (lcsLengths[s1 - 1, s2] if s1 > 0 else 0), (lcsLengths[s1, s2 - 1] if s2 > 0 else 0)
            ext_score = -1
            if sequence1[s1] == sequence2[s2]:
                ext_score = (lcsLengths[s1 - 1, s2 - 1] if s1 > 0 and s2 > 0 else 0) + equalityScore(sequence1[s1], sequence2[s2])
                if s1 > 0 and s2 > 0 and directions[s1 - 1, s2 - 1] == 3: ext_score += lengths[s1 - 1, s2 - 1]
            newValue = max(h_len, v_len, ext_score)
            if newValue == ext_score: lengths[s1, s2], directions[s1, s2] = (lengths[s1 - 1, s2 - 1] if s1 > 0 and s2 > 0 else 0) + 1, 3
            elif newValue == h_len: lengths[s1, s2], directions[s1, s2] = 0, 1
            else: lengths[s1, s2], directions[s1, s2] = 0, 2
            lcsLengths[s1, s2] = newValue
    result, s1, s2, last_s1, last_s2 = [], len(sequence1) - 1, len(sequence2) - 1, len(sequence1), len(sequence2)
    while s1 >= 0 and s2 >= 0:
        if directions[s1, s2] == 3:
            if s1 + 1 != last_s1 or s2 + 1 != last_s2: result.append(SequenceDiff(OffsetRange(s1 + 1, last_s1), OffsetRange(s2 + 1, last_s2)))
            last_s1, last_s2 = s1, s2
            s1, s2 = s1 - 1, s2 - 1
        elif directions[s1, s2] == 1: s1 -= 1
        else: s2 -= 1
    if -1 + 1 != last_s1 or -1 + 1 != last_s2: result.append(SequenceDiff(OffsetRange(0, last_s1), OffsetRange(0, last_s2)))
    result.reverse()
    return result

def get_edit_sequence(before: List[str], after: List[str]) -> tuple[List[str], List[str], List[str]]:
    ess, seq, before_padded, after_padded, b_trav, a_trav = compute(before, after), [], [], [], 0, 0
    for es in ess:
        b_start, b_end, a_start, a_end, b_len, a_len = es.seq1Range.start, es.seq1Range.end, es.seq2Range.start, es.seq2Range.end, len(es.seq1Range), len(es.seq2Range)
        seq.extend([edit_seq_tokens["eql"]] * (b_start - b_trav)); before_padded.extend(before[b_trav:b_start]); after_padded.extend(after[a_trav:a_start])
        if b_len == 0: seq.extend([edit_seq_tokens["add"]] * a_len); before_padded.extend([edit_seq_tokens["padding"]] * a_len); after_padded.extend(after[a_start:a_end])
        elif a_len == 0: seq.extend([edit_seq_tokens["del"]] * b_len); before_padded.extend(before[b_start:b_end]); after_padded.extend([edit_seq_tokens["padding"]] * b_len)
        else:
            if b_len > a_len: seq.extend([edit_seq_tokens["rep"]] * a_len + [edit_seq_tokens["del"]] * (b_len - a_len)); before_padded.extend(before[b_start:b_end]); after_padded.extend(after[a_start:a_end] + [edit_seq_tokens["padding"]] * (b_len - a_len))
            else: seq.extend([edit_seq_tokens["rep"]] * b_len + [edit_seq_tokens["add"]] * (a_len - b_len)); after_padded.extend(after[a_start:a_end]); before_padded.extend(before[b_start:b_end] + [edit_seq_tokens["padding"]] * (a_len - b_len))
        b_trav, a_trav = b_end, a_end
    seq.extend([edit_seq_tokens["eql"]] * (len(before) - b_trav)); before_padded.extend(before[b_trav:]); after_padded.extend(after[a_trav:])
    return seq, before_padded, after_padded


# ==================== 数据集定义 ====================
class MergeDataset(Dataset):
    """Custom PyTorch Dataset for the merge conflict data."""
    def __init__(self, file_path, tokenizer, max_length=MAX_LENGTH, limit=None):
        with open(file_path, 'r', encoding='utf-8') as f:
            self.samples = json.load(f)
        if limit:
            self.samples = self.samples[:limit]
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.samples)

    def _pad_and_truncate(self, token_ids: List[int]) -> List[int]:
        if len(token_ids) > self.max_length:
            return token_ids[:self.max_length]
        else:
            return token_ids + [self.tokenizer.pad_token_id] * (self.max_length - len(token_ids))

    def __getitem__(self, idx):
        sample = self.samples[idx]

        # Expecting raw text in the JSON for 'a', 'b', 'o'
        a_text = sample.get('a_text', '')
        b_text = sample.get('b_text', '')
        o_text = sample.get('o_text', '')

        # Tokenize raw texts using the CodeBERTa tokenizer
        a_tokens = self.tokenizer.tokenize(a_text)
        b_tokens = self.tokenizer.tokenize(b_text)
        o_tokens = self.tokenizer.tokenize(o_text)

        # Generate four edit sequences (list of string tokens) as per MergeBERT paper:
        # E_AO: Edit operations from O to A
        # E_OA: Edit operations from A to O
        # E_BO: Edit operations from O to B
        # E_OB: Edit operations from B to O
        ao_edit_seq, _, _ = get_edit_sequence(o_tokens, a_tokens) # O to A
        oa_edit_seq, _, _ = get_edit_sequence(a_tokens, o_tokens) # A to O
        bo_edit_seq, _, _ = get_edit_sequence(o_tokens, b_tokens) # O to B
        ob_edit_seq, _, _ = get_edit_sequence(b_tokens, o_tokens) # B to O

        # Convert edit sequence tokens (strings like '=', '+') to IDs and pad
        # Note: We assume these special edit tokens are recognized by the tokenizer
        # or converted to UNK which is handled during training.
        a_o_ids = self._pad_and_truncate(self.tokenizer.convert_tokens_to_ids(ao_edit_seq))
        o_a_ids = self._pad_and_truncate(self.tokenizer.convert_tokens_to_ids(oa_edit_seq))
        b_o_ids = self._pad_and_truncate(self.tokenizer.convert_tokens_to_ids(bo_edit_seq))
        o_b_ids = self._pad_and_truncate(self.tokenizer.convert_tokens_to_ids(ob_edit_seq))

        return {
            "a_o_ids": torch.tensor(a_o_ids, dtype=torch.long),
            "o_a_ids": torch.tensor(o_a_ids, dtype=torch.long),
            "b_o_ids": torch.tensor(b_o_ids, dtype=torch.long),
            "o_b_ids": torch.tensor(o_b_ids, dtype=torch.long),
            "labels": torch.tensor(sample['numeric_label'], dtype=torch.long),
            "descriptive_label": sample['descriptive_label']
        }


# ==================== 评估指标 ====================
def compute_metrics(preds, labels):
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted', zero_division=0)
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }


# ==================== 训练函数 ====================
def train_epoch(model, data_loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    scaler = torch.cuda.amp.GradScaler() # Initialize GradScaler for mixed precision

    for batch in tqdm(data_loader, desc="Training"):
        optimizer.zero_grad()

        a_o_ids = batch['a_o_ids'].to(device)
        o_a_ids = batch['o_a_ids'].to(device)
        b_o_ids = batch['b_o_ids'].to(device)
        o_b_ids = batch['o_b_ids'].to(device)
        labels = batch['labels'].to(device)

        with torch.cuda.amp.autocast(): # Enable autocast for mixed precision
            outputs = model(a_o_ids, o_a_ids, b_o_ids, o_b_ids, labels=labels)
            loss = outputs['loss']
            if device.type == 'cuda' and torch.cuda.device_count() > 1:
                loss = loss.mean()

        scaler.scale(loss).backward() # Scale loss and perform backward pass
        scaler.step(optimizer) # Update optimizer
        scaler.update() # Update scaler for next iteration
        scheduler.step()

        total_loss += loss.item()

    return total_loss / len(data_loader)


# ==================== 评估函数 ====================
def eval_epoch(model, data_loader, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_numeric_labels = []
    all_descriptive_labels = []

    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Evaluating"):
            a_o_ids = batch['a_o_ids'].to(device)
            o_a_ids = batch['o_a_ids'].to(device)
            b_o_ids = batch['b_o_ids'].to(device)
            o_b_ids = batch['o_b_ids'].to(device)
            labels = batch['labels'].to(device)
            descriptive_labels = batch['descriptive_label']

            with torch.cuda.amp.autocast(): # Enable autocast for mixed precision during evaluation
                outputs = model(a_o_ids, o_a_ids, b_o_ids, o_b_ids, labels=labels)
                loss = outputs['loss']
            if device.type == 'cuda' and torch.cuda.device_count() > 1:
                loss = loss.mean()
            logits = outputs['logits']

            total_loss += loss.item()
            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_numeric_labels.extend(labels.cpu().numpy())
            all_descriptive_labels.extend(descriptive_labels)

    avg_loss = total_loss / len(data_loader)

    # --- Detailed Analysis ---
    results_by_label = defaultdict(lambda: {"true": [], "pred": []})
    for true_label, pred_label, desc_label in zip(all_numeric_labels, all_preds, all_descriptive_labels):
        results_by_label[desc_label]["true"].append(true_label)
        results_by_label[desc_label]["pred"].append(pred_label)

    logging.info("\n" + "="*20 + " DETAILED EVALUATION PER CATEGORY " + "="*20 + "\n")

    for desc_label, data in sorted(results_by_label.items()):
        y_true = data["true"]
        y_pred = data["pred"]

        if not y_true:
            continue

        logging.info(f"--- Classification Report for Category: '{desc_label}' (count: {len(y_true)}) ---")

        all_numeric_labels_in_group = sorted(list(set(y_true) | set(y_pred)))
        target_names = [REVERSE_LABEL_MAP.get(l, f"unknown_label_{l}") for l in all_numeric_labels_in_group]

        try:
            report = classification_report(
                y_true,
                y_pred,
                labels=all_numeric_labels_in_group,
                target_names=target_names,
                zero_division=0
            )
            logging.info("\n" + report)
        except Exception as e:
            logging.warning(f"Could not generate classification report for '{desc_label}': {e}")
            acc = accuracy_score(y_true, y_pred)
            logging.info(f"  - Overall Accuracy for this group: {acc:.4f}")

        logging.info("--------------------------------------------------------------------\n")

    # Overall metrics
    metrics = compute_metrics(all_preds, all_numeric_labels)
    return avg_loss, metrics


# ==================== 主函数 ====================
def main():
    """Main function to run the training."""
    # 创建保存模型的目录
    os.makedirs(SAVED_MODEL_DIR, exist_ok=True)

    log_filename = f"training_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s [%(levelname)s] %(message)s",
        handlers=[
            logging.FileHandler(log_filename),
            logging.StreamHandler()
        ]
    )
    logging.info("\n\n" + "="*50 + " NEW TRAINING RUN " + "="*50 + "\n")

    # 检测设备 - Colab 优先使用 CUDA
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    logging.info(f"Using device: {device}")

    if device.type == 'cuda':
        logging.info(f"GPU: {torch.cuda.get_device_name(0)}")
        logging.info(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

    # Initialize tokenizer once for the entire training run
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
    logging.info(f"Loaded tokenizer from: {MODEL_PATH}")

    # 数据集名称 (与预处理输出的文件名匹配)
    dataset_name = "data"
    train_file = os.path.join(DATA_DIR, f"{dataset_name}_train.json")
    val_file = os.path.join(DATA_DIR, f"{dataset_name}_validation.json")

    # 检查数据文件是否存在
    if not os.path.exists(train_file):
        raise FileNotFoundError(f"训练数据文件不存在: {train_file}\n请上传 {dataset_name}_train.json 到 {DATA_DIR}. 文件应包含 'a_text', 'b_text', 'o_text', 'numeric_label', 'descriptive_label' 字段。")
    if not os.path.exists(val_file):
        raise FileNotFoundError(f"验证数据文件不存在: {val_file}\n请上传 {dataset_name}_validation.json 到 {DATA_DIR}. 文件应包含 'a_text', 'b_text', 'o_text', 'numeric_label', 'descriptive_label' 字段。")

    train_dataset = MergeDataset(train_file, tokenizer, max_length=MAX_LENGTH)
    val_dataset = MergeDataset(val_file, tokenizer, max_length=MAX_LENGTH)

    logging.info(f"Training samples: {len(train_dataset)}")
    logging.info(f"Validation samples: {len(val_dataset)}")

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, num_workers=2, pin_memory=True)

    model = MergeBERTModel(encoder_path=MODEL_PATH, num_labels=NUM_LABELS)
    # Clear CUDA cache before moving the model to GPU
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    model.to(device)

    if torch.cuda.device_count() > 1:
        logging.info(f"Using {torch.cuda.device_count()} GPUs!")
        model = nn.DataParallel(model)

    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

    best_val_accuracy = 0

    for epoch in range(EPOCHS):
        logging.info(f"\nEpoch {epoch + 1}/{EPOCHS}")
        logging.info("-" * 30)

        train_loss = train_epoch(model, train_loader, optimizer, scheduler, device)
        logging.info(f"Train loss: {train_loss:.4f}")

        val_loss, metrics = eval_epoch(model, val_loader, device)
        logging.info(f"Validation loss: {val_loss:.4f}")
        logging.info(f"Overall validation metrics: {metrics}")

        if metrics['accuracy'] > best_val_accuracy:
            best_val_accuracy = metrics['accuracy']
            model_to_save = model.module if hasattr(model, 'module') else model
            save_path = os.path.join(SAVED_MODEL_DIR, 'mergebert.pt')
            torch.save(model_to_save.state_dict(), save_path)
            logging.info(f"\u2713 Best model saved to {save_path}")

    logging.info("\n" + "="*50)
    logging.info(f"Training completed! Best validation accuracy: {best_val_accuracy:.4f}")
    logging.info(f"Model saved to: {SAVED_MODEL_DIR}mergebert.pt")
    logging.info("="*50)


if __name__ == "__main__":
    main()


In [ ]:
"""
MergeBERT Inference Script for Google Colab

使用方法:
1. 上传以下文件到 Colab 的 /content/ 目录:
   - inferencecolab.py (本文件)
   - mergebert.pt (训练好的模型权重)

2. 在 Colab 中运行以下命令安装依赖:
   !pip install torch transformers GitPython

3. 准备冲突文件 (三个版本):
   - file_a.txt (你的版本)
   - file_b.txt (他们的版本)
   - file_o.txt (共同祖先/基础版本)

4. 运行推理:
   方式1: 命令行
   !python inferencecolab.py --file_a file_a.txt --file_b file_b.txt --file_o file_o.txt

   方式2: 在 notebook 中调用函数
   from inferencecolab import resolve_conflict
   result = resolve_conflict("代码A内容", "代码B内容", "基础版本内容")
   print(result)
"""

import torch
import argparse
import os
import re
import shutil
import numpy as np
from pathlib import Path
from typing import Callable, List
from transformers import AutoTokenizer, RobertaModel, RobertaConfig
import torch.nn as nn

# ==================== Colab 路径配置 ====================
# 使用 Hugging Face 在线模型
MODEL_PATH = "huggingface/CodeBERTa-small-v1"
# 训练好的模型权重路径
SAVED_MODEL_PATH = "/content/saved_models/mergebert.pt"
MAX_LENGTH = 512
NUM_LABELS = 9

# 临时 git 仓库路径
TEMP_REPO_BASE_PATH = Path("/tmp/mergebert_repo_temp")

# ==================== 标签映射 ====================
REVERSE_LABEL_MAP = {
    0: "select_a",
    1: "select_b",
    2: "select_o",
    3: "concat_ab",
    4: "concat_ba",
    5: "select_a_del_o",
    6: "select_b_del_o",
    7: "concat_ab_del_o",
    8: "concat_ba_del_o",
}


# ==================== 模型定义 (内嵌 model.py) ====================
class MergeBERTModel(nn.Module):
    """MergeBERT model for merge conflict resolution."""
    def __init__(self, encoder_path, num_labels=9):
        super(MergeBERTModel, self).__init__()
        self.num_labels = num_labels

        config = RobertaConfig.from_pretrained(encoder_path)
        self.encoder = RobertaModel.from_pretrained(encoder_path, config=config)

        # Aggregation weights (learnable)
        self.aggregation_weights = nn.Parameter(torch.ones(4))

        # Classification head
        self.classifier = nn.Linear(config.hidden_size, num_labels)

    def forward(self, a_o_ids, o_a_ids, b_o_ids, o_b_ids, labels=None):
        output_ao = self.encoder(input_ids=a_o_ids).pooler_output
        output_oa = self.encoder(input_ids=o_a_ids).pooler_output
        output_bo = self.encoder(input_ids=b_o_ids).pooler_output
        output_ob = self.encoder(input_ids=o_b_ids).pooler_output

        agg_weights = nn.functional.softmax(self.aggregation_weights, dim=0)

        aggregated_output = (
            agg_weights[0] * output_ao +
            agg_weights[1] * output_oa +
            agg_weights[2] * output_bo +
            agg_weights[3] * output_ob
        )

        logits = self.classifier(aggregated_output)

        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))

        return {
            "loss": loss,
            "logits": logits
        }


# ==================== 工具函数 (内嵌 utils.py 关键部分) ====================
edit_seq_tokens = {
    "eql": "=", "add": "+", "del": "-", "rep": "↔", "padding": "[PAD]"
}

def normalize_code(text: str) -> str:
    """Removes indents, empty lines, and duplicate whitespace."""
    lines = text.split('\n')
    processed_lines = [line.strip() for line in lines if line.strip()]
    return re.sub(r'\s+', ' ', " ".join(processed_lines))

def apply_pattern(pattern, a, b, o):
    if pattern == "select_a": return a
    if pattern == "select_b": return b
    if pattern == "select_o": return o
    if pattern == "concat_ab": return a + b
    if pattern == "concat_ba": return b + a
    o_set = set(o)
    if pattern == "select_a_del_o": return [token for token in a if token not in o_set]
    if pattern == "select_b_del_o": return [token for token in b if token not in o_set]
    if pattern == "concat_ab_del_o": return [token for token in a + b if token not in o_set]
    if pattern == "concat_ba_del_o": return [token for token in b + a if token not in o_set]
    return None

class OffsetRange:
    def __init__(self, start: int, end: int):
        self.start, self.end = start, end
    def __len__(self): return self.end - self.start

class SequenceDiff:
    def __init__(self, seq1Range: OffsetRange, seq2Range: OffsetRange):
        self.seq1Range, self.seq2Range = seq1Range, seq2Range

def compute(sequence1: List[str], sequence2: List[str], equalityScore: Callable[[int, int], float] = lambda x, y: 1 if x == y else 0) -> List[SequenceDiff]:
    if not sequence1 or not sequence2: return [SequenceDiff(OffsetRange(0, len(sequence1)), OffsetRange(0, len(sequence2)))]
    lcsLengths, directions, lengths = np.zeros((len(sequence1), len(sequence2))), np.zeros((len(sequence1), len(sequence2))), np.zeros((len(sequence1), len(sequence2)))
    for s1 in range(len(sequence1)):
        for s2 in range(len(sequence2)):
            h_len, v_len = (lcsLengths[s1 - 1, s2] if s1 > 0 else 0), (lcsLengths[s1, s2 - 1] if s2 > 0 else 0)
            ext_score = -1
            if sequence1[s1] == sequence2[s2]:
                ext_score = (lcsLengths[s1 - 1, s2 - 1] if s1 > 0 and s2 > 0 else 0) + equalityScore(sequence1[s1], sequence2[s2])
                if s1 > 0 and s2 > 0 and directions[s1 - 1, s2 - 1] == 3: ext_score += lengths[s1 - 1, s2 - 1]
            newValue = max(h_len, v_len, ext_score)
            if newValue == ext_score: lengths[s1, s2], directions[s1, s2] = (lengths[s1 - 1, s2 - 1] if s1 > 0 and s2 > 0 else 0) + 1, 3
            elif newValue == h_len: lengths[s1, s2], directions[s1, s2] = 0, 1
            else: lengths[s1, s2], directions[s1, s2] = 0, 2
            lcsLengths[s1, s2] = newValue
    result, s1, s2, last_s1, last_s2 = [], len(sequence1) - 1, len(sequence2) - 1, len(sequence1), len(sequence2)
    while s1 >= 0 and s2 >= 0:
        if directions[s1, s2] == 3:
            if s1 + 1 != last_s1 or s2 + 1 != last_s2: result.append(SequenceDiff(OffsetRange(s1 + 1, last_s1), OffsetRange(s2 + 1, last_s2)))
            last_s1, last_s2 = s1, s2
            s1, s2 = s1 - 1, s2 - 1
        elif directions[s1, s2] == 1: s1 -= 1
        else: s2 -= 1
    if -1 + 1 != last_s1 or -1 + 1 != last_s2: result.append(SequenceDiff(OffsetRange(0, last_s1), OffsetRange(0, last_s2)))
    result.reverse()
    return result

def get_edit_sequence(before: List[str], after: List[str]) -> List[List[str]]:
    ess, seq, before_padded, after_padded, b_trav, a_trav = compute(before, after), [], [], [], 0, 0
    for es in ess:
        b_start, b_end, a_start, a_end, b_len, a_len = es.seq1Range.start, es.seq1Range.end, es.seq2Range.start, es.seq2Range.end, len(es.seq1Range), len(es.seq2Range)
        seq.extend([edit_seq_tokens["eql"]] * (b_start - b_trav)); before_padded.extend(before[b_trav:b_start]); after_padded.extend(after[a_trav:a_start])
        if b_len == 0: seq.extend([edit_seq_tokens["add"]] * a_len); before_padded.extend([edit_seq_tokens["padding"]] * a_len); after_padded.extend(after[a_start:a_end])
        elif a_len == 0: seq.extend([edit_seq_tokens["del"]] * b_len); before_padded.extend(before[b_start:b_end]); after_padded.extend([edit_seq_tokens["padding"]] * b_len)
        else:
            if b_len > a_len: seq.extend([edit_seq_tokens["rep"]] * a_len + [edit_seq_tokens["del"]] * (b_len - a_len)); before_padded.extend(before[b_start:b_end]); after_padded.extend(after[a_start:a_end] + [edit_seq_tokens["padding"]] * (b_len - a_len))
            else: seq.extend([edit_seq_tokens["rep"]] * b_len + [edit_seq_tokens["add"]] * (a_len - b_len)); after_padded.extend(after[a_start:a_end]); before_padded.extend(before[b_start:b_end] + [edit_seq_tokens["padding"]] * (a_len - b_len))
        b_trav, a_trav = b_end, a_end
    seq.extend([edit_seq_tokens["eql"]] * (len(before) - b_trav)); before_padded.extend(before[b_trav:]); after_padded.extend(after[a_trav:])
    return seq, before_padded, after_padded

def align_and_get_edit_sequence(before: List[str], after: List[str]) -> List[List[str]]:
    return get_edit_sequence(before, after)

def write_tokens_to_file(tokens, file_path):
    with open(file_path, 'w', encoding='utf-8') as f:
        for token in tokens: f.write(token + '\n')

def read_tokens_from_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as f: return [line.strip('\n') for line in f.readlines()]

def token_level_merge(a_tokens, b_tokens, o_tokens):
    try:
        from git import Repo, Git
    except ImportError:
        raise ImportError("请先安装 GitPython: !pip install GitPython")

    temp_repo_path = TEMP_REPO_BASE_PATH / str(os.getpid())
    if temp_repo_path.exists(): shutil.rmtree(temp_repo_path)
    repo = Repo.init(temp_repo_path, initial_branch='master'); _git = Git(temp_repo_path); _git.config("merge.conflictstyle", "diff3"); tmp_file = temp_repo_path / "tokens.txt"
    write_tokens_to_file(o_tokens, tmp_file); _git.add('.'); repo.index.commit("base")
    _git.branch('a_branch'); _git.checkout('a_branch'); write_tokens_to_file(a_tokens, tmp_file); _git.add('.'); repo.index.commit("commit a")
    _git.checkout('master'); _git.branch('b_branch'); _git.checkout('b_branch'); write_tokens_to_file(b_tokens, tmp_file); _git.add('.'); repo.index.commit("commit b")
    try: _git.merge('a_branch'); return [read_tokens_from_file(tmp_file)]
    except Exception: return parse_conflict_markers(tmp_file.read_text())

def parse_conflict_markers(content):
    hunks, lines, current_hunk, i = [], content.split('\n'), [], 0
    while i < len(lines):
        line = lines[i]
        if line.startswith('<<<<<<<'):
            if current_hunk: hunks.append([h for h in current_hunk if h]); current_hunk = []
            a_conflict, o_conflict, b_conflict = [], [], []; i += 1
            while not lines[i].startswith('|||||||'): a_conflict.append(lines[i]); i += 1
            i += 1
            while not lines[i].startswith('======='): o_conflict.append(lines[i]); i += 1
            i += 1
            while not lines[i].startswith('>>>>>>>'): b_conflict.append(lines[i]); i += 1
            hunks.append((a_conflict, b_conflict, o_conflict))
        else: current_hunk.append(line)
        i += 1
    if current_hunk: current_hunk = [h for h in current_hunk if h];
    if current_hunk: hunks.append(current_hunk)
    return hunks


# ==================== 全局变量 (用于缓存模型) ====================
_cached_model = None
_cached_tokenizer = None
_cached_device = None


def load_model(model_path=MODEL_PATH, checkpoint_path=SAVED_MODEL_PATH, device=None):
    """加载模型 (支持缓存，避免重复加载)"""
    global _cached_model, _cached_tokenizer, _cached_device

    if _cached_model is not None:
        return _cached_model, _cached_tokenizer, _cached_device

    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print(f"Using device: {device}")
    print(f"Loading tokenizer from: {model_path}")
    tokenizer = AutoTokenizer.from_pretrained(model_path)

    print(f"Loading model from: {model_path}")
    print(f"Loading checkpoint from: {checkpoint_path}")
    model = MergeBERTModel(encoder_path=model_path, num_labels=NUM_LABELS)
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model.to(device)
    model.eval()

    _cached_model = model
    _cached_tokenizer = tokenizer
    _cached_device = device

    return model, tokenizer, device


def resolve_conflict(a_text: str, b_text: str, o_text: str,
                     model_path=MODEL_PATH,
                     checkpoint_path=SAVED_MODEL_PATH) -> str:
    """
    解决合并冲突的主函数

    参数:
        a_text: 版本 A 的代码 (你的版本)
        b_text: 版本 B 的代码 (他们的版本)
        o_text: 基础版本的代码 (共同祖先)
        model_path: 预训练模型路径
        checkpoint_path: 训练好的权重路径

    返回:
        resolved_text: 解决冲突后的代码
    """
    model, tokenizer, device = load_model(model_path, checkpoint_path)

    # Tokenize
    a_tokens = tokenizer.tokenize(a_text)
    b_tokens = tokenizer.tokenize(b_text)
    o_tokens = tokenizer.tokenize(o_text)

    # Token-level merge
    hunks = token_level_merge(a_tokens, b_tokens, o_tokens)
    conflict_hunks = [h for h in hunks if isinstance(h, tuple)]

    if len(conflict_hunks) == 0:
        print("No conflict detected at token level. Returning merged result.")
        merged_tokens = hunks[0] if hunks else []
        return tokenizer.convert_tokens_to_string(merged_tokens)

    if len(conflict_hunks) > 1:
        raise ValueError(f"Found {len(conflict_hunks)} conflicts. This model only supports single-conflict resolution.")

    # Find conflict index
    conflict_idx = -1
    for i, hunk in enumerate(hunks):
        if isinstance(hunk, tuple):
            conflict_idx = i
            break

    prefix_tokens = [token for hunk in hunks[:conflict_idx] for token in hunk]
    suffix_tokens = [token for hunk in hunks[conflict_idx+1:] for token in hunk]
    a_conflict, b_conflict, o_conflict = conflict_hunks[0]

    # Prepare data for model
    a_o_aligned, o_a_aligned, _ = align_and_get_edit_sequence(a_tokens, o_tokens)
    b_o_aligned, o_b_aligned, _ = align_and_get_edit_sequence(b_tokens, o_tokens)

    def process_sequence(seq):
        ids = tokenizer.convert_tokens_to_ids(seq)
        if len(ids) > MAX_LENGTH:
            ids = ids[:MAX_LENGTH]
        else:
            ids += [tokenizer.pad_token_id] * (MAX_LENGTH - len(ids))
        return torch.tensor(ids, dtype=torch.long).unsqueeze(0).to(device)

    a_o_ids = process_sequence(a_o_aligned)
    o_a_ids = process_sequence(o_a_aligned)
    b_o_ids = process_sequence(b_o_aligned)
    o_b_ids = process_sequence(o_b_aligned)

    # Run inference
    with torch.no_grad():
        outputs = model(a_o_ids, o_a_ids, b_o_ids, o_b_ids)
        logits = outputs['logits']
        prediction = torch.argmax(logits, dim=1).item()

    # Decode and reconstruct
    predicted_pattern = REVERSE_LABEL_MAP.get(prediction)
    if not predicted_pattern:
        raise ValueError(f"Unknown pattern predicted with ID: {prediction}")

    print(f"Predicted resolution pattern: {predicted_pattern}")

    resolved_conflict = apply_pattern(predicted_pattern, a_conflict, b_conflict, o_conflict)
    reconstructed_tokens = prefix_tokens + resolved_conflict + suffix_tokens
    resolved_text = tokenizer.convert_tokens_to_string(reconstructed_tokens)

    return resolved_text


def main():
    """命令行入口"""
    parser = argparse.ArgumentParser(description="MergeBERT Inference for Colab")
    parser.add_argument("--file_a", type=str, required=True, help="Path to file A (yours)")
    parser.add_argument("--file_b", type=str, required=True, help="Path to file B (theirs)")
    parser.add_argument("--file_o", type=str, required=True, help="Path to file O (base)")
    parser.add_argument("--model_path", type=str, default=MODEL_PATH,
                        help="Path to pretrained CodeBERTa model")
    parser.add_argument("--checkpoint", type=str, default=SAVED_MODEL_PATH,
                        help="Path to trained mergebert.pt checkpoint")
    parser.add_argument("--output", type=str, default=None,
                        help="Path to save resolved output (optional)")
    args = parser.parse_args()

    # 检查文件是否存在
    for path, name in [(args.file_a, "A"), (args.file_b, "B"), (args.file_o, "O")]:
        if not os.path.exists(path):
            raise FileNotFoundError(f"File {name} not found: {path}")

    if not os.path.exists(args.checkpoint):
        raise FileNotFoundError(f"Model checkpoint not found: {args.checkpoint}\n"
                                f"Please upload mergebert.pt to {args.checkpoint}")

    # Read file contents
    with open(args.file_a, 'r', encoding='utf-8') as f:
        a_text = f.read()
    with open(args.file_b, 'r', encoding='utf-8') as f:
        b_text = f.read()
    with open(args.file_o, 'r', encoding='utf-8') as f:
        o_text = f.read()

    # Resolve conflict
    resolved_text = resolve_conflict(a_text, b_text, o_text,
                                     args.model_path, args.checkpoint)

    print("\n" + "="*50)
    print("--- Resolved Code ---")
    print("="*50)
    print(resolved_text)

    # Optionally save to file
    if args.output:
        with open(args.output, 'w', encoding='utf-8') as f:
            f.write(resolved_text)
        print(f"\n✓ Resolved code saved to: {args.output}")


def is_notebook():
    """检测是否在 Jupyter/Colab notebook 环境中运行"""
    try:
        from IPython import get_ipython
        if get_ipython() is not None:
            return True
    except:
        pass
    return False


if __name__ == "__main__":
    # 在 notebook 环境中不自动运行 main()
    # 用户应该使用 resolve_conflict() 函数
    if not is_notebook():
        main()
    else:
        print("="*60)
        print("在 Colab/Jupyter Notebook 中运行推理，请使用以下方式：")
        print("="*60)
        print("""
# 方式 1: 直接传入代码字符串
from inferencecolab import resolve_conflict

a_code = '''你的版本代码'''
b_code = '''他们的版本代码'''
o_code = '''基础版本代码'''

result = resolve_conflict(a_code, b_code, o_code)
print(result)

# 方式 2: 从文件读取
with open('file_a.txt') as f: a_code = f.read()
with open('file_b.txt') as f: b_code = f.read()
with open('file_o.txt') as f: o_code = f.read()

result = resolve_conflict(a_code, b_code, o_code)
print(result)
""")



In [ ]:
a_code = "def hello(): print('A')"
b_code = "def hello(): print('B')"
o_code = "def hello(): pass"

result = resolve_conflict(a_code, b_code, o_code)
print(result)

# 上传需要测试的数据集json文件，根据文件名修改 INPUT_FILE

In [ ]:
"""
MergeBERT Evaluation Script for Google Colab

使用方法:
1. 确保已上传以下文件到 Colab:
   - inferencecolab.py
   - mergebert.pt (训练好的模型权重)
   - autopsy_time20.json (测试数据)

2. 运行:
   !python eval.py

或在 notebook 中:
   %run eval.py
"""

import json
import os
import torch
import numpy as np
from tqdm import tqdm
from typing import List, Dict



# ==================== 配置 ====================
INPUT_FILE = "/content/spring-boot_time20.json"
OUTPUT_FILE = "/content/sis_time20eval.json"
TOP_K = 5  # 获取 top5 预测结果


def load_model_for_eval(model_path=MODEL_PATH, checkpoint_path=SAVED_MODEL_PATH):
    """加载模型用于评估"""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    tokenizer = AutoTokenizer.from_pretrained(model_path)
    print(f"Loaded tokenizer from: {model_path}")

    model = MergeBERTModel(encoder_path=model_path, num_labels=NUM_LABELS)
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model.to(device)
    model.eval()
    print(f"Loaded model checkpoint from: {checkpoint_path}")

    return model, tokenizer, device


def get_topk_predictions(model, tokenizer, device, a_text, b_text, o_text, k=5):
    """
    获取 top-k 预测结果

    返回:
        List[str]: top-k 个预测的解析结果文本
    """
    try:
        # Tokenize
        a_tokens = tokenizer.tokenize(a_text) if a_text else []
        b_tokens = tokenizer.tokenize(b_text) if b_text else []
        o_tokens = tokenizer.tokenize(o_text) if o_text else []

        # 如果所有内容都为空，返回空结果
        if not a_tokens and not b_tokens and not o_tokens:
            return [""] * k

        # Token-level merge
        hunks = token_level_merge(a_tokens, b_tokens, o_tokens)
        conflict_hunks = [h for h in hunks if isinstance(h, tuple)]

        # 如果没有冲突，直接返回合并结果
        if len(conflict_hunks) == 0:
            merged_tokens = hunks[0] if hunks else []
            merged_text = tokenizer.convert_tokens_to_string(merged_tokens)
            return [merged_text] * k  # 返回相同结果

        # 如果有多个冲突，暂时只处理第一个
        conflict_idx = -1
        for i, hunk in enumerate(hunks):
            if isinstance(hunk, tuple):
                conflict_idx = i
                break

        prefix_tokens = [token for hunk in hunks[:conflict_idx] for token in hunk]
        suffix_tokens = [token for hunk in hunks[conflict_idx+1:] for token in hunk]
        a_conflict, b_conflict, o_conflict = conflict_hunks[0]

        # Prepare data for model
        a_o_aligned, o_a_aligned, _ = align_and_get_edit_sequence(a_tokens, o_tokens)
        b_o_aligned, o_b_aligned, _ = align_and_get_edit_sequence(b_tokens, o_tokens)

        def process_sequence(seq):
            ids = tokenizer.convert_tokens_to_ids(seq)
            if len(ids) > MAX_LENGTH:
                ids = ids[:MAX_LENGTH]
            else:
                ids += [tokenizer.pad_token_id] * (MAX_LENGTH - len(ids))
            return torch.tensor(ids, dtype=torch.long).unsqueeze(0).to(device)

        a_o_ids = process_sequence(a_o_aligned)
        o_a_ids = process_sequence(o_a_aligned)
        b_o_ids = process_sequence(b_o_aligned)
        o_b_ids = process_sequence(o_b_aligned)

        # Run inference and get top-k predictions
        with torch.no_grad():
            outputs = model(a_o_ids, o_a_ids, b_o_ids, o_b_ids)
            logits = outputs['logits']

            # Get top-k predictions
            probs = torch.softmax(logits, dim=1)
            topk_probs, topk_indices = torch.topk(probs, min(k, NUM_LABELS), dim=1)
            topk_indices = topk_indices[0].cpu().numpy()

        # Generate resolved text for each top-k prediction
        results = []
        for pred_idx in topk_indices:
            pattern = REVERSE_LABEL_MAP.get(int(pred_idx))
            if pattern:
                resolved_conflict = apply_pattern(pattern, a_conflict, b_conflict, o_conflict)
                if resolved_conflict is not None:
                    reconstructed_tokens = prefix_tokens + resolved_conflict + suffix_tokens
                    resolved_text = tokenizer.convert_tokens_to_string(reconstructed_tokens)
                    results.append(resolved_text)
                else:
                    results.append("")
            else:
                results.append("")

        # Pad to k results if needed
        while len(results) < k:
            results.append("")

        return results[:k]

    except Exception as e:
        print(f"Error during prediction: {e}")
        return [""] * k


def evaluate_dataset(input_file, output_file, model_path=MODEL_PATH, checkpoint_path=SAVED_MODEL_PATH):
    """评估整个数据集"""
    # 加载数据
    print(f"Loading data from: {input_file}")
    with open(input_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    print(f"Loaded {len(data)} samples")

    # 加载模型
    model, tokenizer, device = load_model_for_eval(model_path, checkpoint_path)

    # 处理每个样本
    results = []
    for item in tqdm(data, desc="Evaluating"):
        # 提取输入
        a_text = item.get("a_contents", "")
        b_text = item.get("b_contents", "")
        o_text = item.get("o_contents", "")

        # 获取 top-k 预测
        answers = get_topk_predictions(model, tokenizer, device, a_text, b_text, o_text, k=TOP_K)

        # 创建结果对象（保留原始字段并添加 answers）
        result_item = item.copy()
        result_item["answers"] = answers
        results.append(result_item)

    # 保存结果
    print(f"Saving results to: {output_file}")
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    print(f"Evaluation complete. Saved {len(results)} results.")

    return results


# ==================== 评估指标函数 ====================
def levenshtein_distance(s1: str, s2: str) -> int:
    """计算两个字符串之间的 Levenshtein 编辑距离"""
    if len(s1) < len(s2):
        return levenshtein_distance(s2, s1)

    if len(s2) == 0:
        return len(s1)

    previous_row = range(len(s2) + 1)
    for i, c1 in enumerate(s1):
        current_row = [i + 1]
        for j, c2 in enumerate(s2):
            # 插入、删除、替换的代价
            insertions = previous_row[j + 1] + 1
            deletions = current_row[j] + 1
            substitutions = previous_row[j] + (c1 != c2)
            current_row.append(min(insertions, deletions, substitutions))
        previous_row = current_row

    return previous_row[-1]


def perfect_match_rate(ours: str, target: str) -> float:
    """
    计算两个字符串的相似度（基于 Levenshtein 距离）

    参数:
        ours: 预测结果
        target: 目标结果

    返回:
        相似度百分比 (0-100)
    """
    import re
    # 移除所有空白字符
    ours = re.sub(r'\s*', '', ours) if ours else ''
    target = re.sub(r'\s*', '', target) if target else ''

    if len(ours) == 0 and len(target) == 0:
        return 100.0

    # 计算 Levenshtein 距离
    distance = levenshtein_distance(ours, target)

    # 计算相似度
    max_len = max(len(ours), len(target))
    similarity = 1 - distance / max_len if max_len > 0 else 1.0

    # 返回百分比形式，保留两位小数
    return round(similarity * 100, 2)


def is_perfect_match(cand: str, resolution: str, threshold: float = 100.0) -> bool:
    """判断是否为完美匹配（相似度达到阈值）"""
    return perfect_match_rate(cand, resolution) >= threshold


def count_topk_perfect_match(data: List[dict], answer_key: str = "answers") -> Dict[str, float]:
    """计算 top-k 完全匹配率"""
    result = {
        "total": len(data),
        "top1_perfect": 0,
        "top3_perfect": 0,
        "top5_perfect": 0
    }

    for obj in data:
        resolution = obj.get("res_region", "").strip()
        answers = obj.get(answer_key, [])

        top1 = top3 = top5 = False

        for k, cand in enumerate(answers[:5]):
            if is_perfect_match(cand, resolution):
                if k == 0:
                    top1 = True
                if k < 3:
                    top3 = True
                top5 = True
                break

        result["top1_perfect"] += int(top1)
        result["top3_perfect"] += int(top3 or top1)
        result["top5_perfect"] += int(top5 or top3 or top1)

    total = result["total"]
    if total > 0:
        result["top1_rate_%"] = result["top1_perfect"] / total * 100
        result["top3_rate_%"] = result["top3_perfect"] / total * 100
        result["top5_rate_%"] = result["top5_perfect"] / total * 100
    else:
        result["top1_rate_%"] = 0
        result["top3_rate_%"] = 0
        result["top5_rate_%"] = 0

    return result


def main():
    """主函数"""
    # 检查输入文件
    if not os.path.exists(INPUT_FILE):
        print(f"Error: Input file not found: {INPUT_FILE}")
        print("Please upload autopsy_time20.json to /content/")
        return

    # 检查模型文件
    if not os.path.exists(SAVED_MODEL_PATH):
        print(f"Error: Model checkpoint not found: {SAVED_MODEL_PATH}")
        print("Please upload mergebert.pt to /content/saved_models/")
        return

    # 运行评估
    results = evaluate_dataset(INPUT_FILE, OUTPUT_FILE)

    # 计算指标
    print("\n" + "="*50)
    print("Evaluation Results:")
    print("="*50)

    metrics = count_topk_perfect_match(results, "answers")

    print(f"Total samples: {metrics['total']}")
    print(f"Top-1 Perfect Match: {metrics['top1_perfect']} ({metrics['top1_rate_%']:.2f}%)")
    print(f"Top-3 Perfect Match: {metrics['top3_perfect']} ({metrics['top3_rate_%']:.2f}%)")
    print(f"Top-5 Perfect Match: {metrics['top5_perfect']} ({metrics['top5_rate_%']:.2f}%)")
    print("="*50)

    return metrics


if __name__ == "__main__":
    main()


# 上传测试数据到test目录下

In [ ]:
"""
MergeBERT Evaluation Script for Google Colab

使用方法:
1. 确保已上传以下文件到 Colab:
   - inferencecolab.py
   - mergebert.pt (训练好的模型权重)
   - autopsy_time20.json (测试数据)

2. 运行:
   !python eval.py

或在 notebook 中:
   %run eval.py
"""

import json
import os
import glob
import torch
import numpy as np
from tqdm import tqdm
from typing import List, Dict

# 从 inferencecolab 导入必要的组件

from transformers import AutoTokenizer

# ==================== 配置 ====================
# 测试数据目录
TEST_DIR = "test/"
TOP_K = 1  # 获取 top5 预测结果


def load_model_for_eval(model_path=MODEL_PATH, checkpoint_path=SAVED_MODEL_PATH):
    """加载模型用于评估"""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    tokenizer = AutoTokenizer.from_pretrained(model_path)
    print(f"Loaded tokenizer from: {model_path}")

    model = MergeBERTModel(encoder_path=model_path, num_labels=NUM_LABELS)
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model.to(device)
    model.eval()
    print(f"Loaded model checkpoint from: {checkpoint_path}")

    return model, tokenizer, device


def get_topk_predictions(model, tokenizer, device, a_text, b_text, o_text, k=5):
    """
    获取 top-k 预测结果

    返回:
        List[str]: top-k 个预测的解析结果文本
    """
    try:
        # Tokenize
        a_tokens = tokenizer.tokenize(a_text) if a_text else []
        b_tokens = tokenizer.tokenize(b_text) if b_text else []
        o_tokens = tokenizer.tokenize(o_text) if o_text else []

        # 如果所有内容都为空，返回空结果
        if not a_tokens and not b_tokens and not o_tokens:
            return [""] * k

        # Token-level merge
        hunks = token_level_merge(a_tokens, b_tokens, o_tokens)
        conflict_hunks = [h for h in hunks if isinstance(h, tuple)]

        # 如果没有冲突，直接返回合并结果
        if len(conflict_hunks) == 0:
            merged_tokens = hunks[0] if hunks else []
            merged_text = tokenizer.convert_tokens_to_string(merged_tokens)
            return [merged_text] * k  # 返回相同结果

        # 如果有多个冲突，暂时只处理第一个
        conflict_idx = -1
        for i, hunk in enumerate(hunks):
            if isinstance(hunk, tuple):
                conflict_idx = i
                break

        prefix_tokens = [token for hunk in hunks[:conflict_idx] for token in hunk]
        suffix_tokens = [token for hunk in hunks[conflict_idx+1:] for token in hunk]
        a_conflict, b_conflict, o_conflict = conflict_hunks[0]

        # Prepare data for model
        a_o_aligned, o_a_aligned, _ = align_and_get_edit_sequence(a_tokens, o_tokens)
        b_o_aligned, o_b_aligned, _ = align_and_get_edit_sequence(b_tokens, o_tokens)

        def process_sequence(seq):
            ids = tokenizer.convert_tokens_to_ids(seq)
            if len(ids) > MAX_LENGTH:
                ids = ids[:MAX_LENGTH]
            else:
                ids += [tokenizer.pad_token_id] * (MAX_LENGTH - len(ids))
            return torch.tensor(ids, dtype=torch.long).unsqueeze(0).to(device)

        a_o_ids = process_sequence(a_o_aligned)
        o_a_ids = process_sequence(o_a_aligned)
        b_o_ids = process_sequence(b_o_aligned)
        o_b_ids = process_sequence(o_b_aligned)

        # Run inference and get top-k predictions
        with torch.no_grad():
            outputs = model(a_o_ids, o_a_ids, b_o_ids, o_b_ids)
            logits = outputs['logits']

            # Get top-k predictions
            probs = torch.softmax(logits, dim=1)
            topk_probs, topk_indices = torch.topk(probs, min(k, NUM_LABELS), dim=1)
            topk_indices = topk_indices[0].cpu().numpy()

        # Generate resolved text for each top-k prediction
        results = []
        for pred_idx in topk_indices:
            pattern = REVERSE_LABEL_MAP.get(int(pred_idx))
            if pattern:
                resolved_conflict = apply_pattern(pattern, a_conflict, b_conflict, o_conflict)
                if resolved_conflict is not None:
                    reconstructed_tokens = prefix_tokens + resolved_conflict + suffix_tokens
                    resolved_text = tokenizer.convert_tokens_to_string(reconstructed_tokens)
                    results.append(resolved_text)
                else:
                    results.append("")
            else:
                results.append("")

        # Pad to k results if needed
        while len(results) < k:
            results.append("")

        return results[:k]

    except Exception as e:
        print(f"Error during prediction: {e}")
        return [""] * k


def evaluate_dataset(input_file, output_file, model_path=MODEL_PATH, checkpoint_path=SAVED_MODEL_PATH):
    """评估整个数据集"""
    # 加载数据
    print(f"Loading data from: {input_file}")
    with open(input_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    print(f"Loaded {len(data)} samples")

    # 加载模型
    model, tokenizer, device = load_model_for_eval(model_path, checkpoint_path)

    # 处理每个样本
    results = []
    for item in tqdm(data, desc="Evaluating"):
        # 提取输入
        a_text = item.get("a_contents", "")
        b_text = item.get("b_contents", "")
        o_text = item.get("o_contents", "")

        # 获取 top-k 预测
        answers = get_topk_predictions(model, tokenizer, device, a_text, b_text, o_text, k=TOP_K)

        # 创建结果对象（保留原始字段并添加 answers）
        result_item = item.copy()
        result_item["answers"] = answers
        results.append(result_item)

    # 保存结果
    print(f"Saving results to: {output_file}")
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    print(f"Evaluation complete. Saved {len(results)} results.")

    return results


# ==================== 评估指标函数 ====================
def levenshtein_distance(s1: str, s2: str) -> int:
    """计算两个字符串之间的 Levenshtein 编辑距离"""
    if len(s1) < len(s2):
        return levenshtein_distance(s2, s1)

    if len(s2) == 0:
        return len(s1)

    previous_row = range(len(s2) + 1)
    for i, c1 in enumerate(s1):
        current_row = [i + 1]
        for j, c2 in enumerate(s2):
            # 插入、删除、替换的代价
            insertions = previous_row[j + 1] + 1
            deletions = current_row[j] + 1
            substitutions = previous_row[j] + (c1 != c2)
            current_row.append(min(insertions, deletions, substitutions))
        previous_row = current_row

    return previous_row[-1]


def perfect_match_rate(ours: str, target: str) -> float:
    """
    计算两个字符串的相似度（基于 Levenshtein 距离）

    参数:
        ours: 预测结果
        target: 目标结果

    返回:
        相似度百分比 (0-100)
    """
    import re
    # 移除所有空白字符
    ours = re.sub(r'\s*', '', ours) if ours else ''
    target = re.sub(r'\s*', '', target) if target else ''

    if len(ours) == 0 and len(target) == 0:
        return 100.0

    # 计算 Levenshtein 距离
    distance = levenshtein_distance(ours, target)

    # 计算相似度
    max_len = max(len(ours), len(target))
    similarity = 1 - distance / max_len if max_len > 0 else 1.0

    # 返回百分比形式，保留两位小数
    return round(similarity * 100, 2)


def is_perfect_match(cand: str, resolution: str, threshold: float = 100.0) -> bool:
    """判断是否为完美匹配（相似度达到阈值）"""
    return perfect_match_rate(cand, resolution) >= threshold


def count_topk_perfect_match(data: List[dict], answer_key: str = "answers") -> Dict[str, float]:
    """计算 top-k 完全匹配率"""
    result = {
        "total": len(data),
        "top1_perfect": 0,
        "top3_perfect": 0,
        "top5_perfect": 0
    }

    for obj in data:
        resolution = obj.get("res_region", "").strip()
        answers = obj.get(answer_key, [])

        top1 = top3 = top5 = False

        for k, cand in enumerate(answers[:5]):
            if is_perfect_match(cand, resolution):
                if k == 0:
                    top1 = True
                if k < 3:
                    top3 = True
                top5 = True
                break

        result["top1_perfect"] += int(top1)
        result["top3_perfect"] += int(top3 or top1)
        result["top5_perfect"] += int(top5 or top3 or top1)

    total = result["total"]
    if total > 0:
        result["top1_rate_%"] = result["top1_perfect"] / total * 100
        result["top3_rate_%"] = result["top3_perfect"] / total * 100
        result["top5_rate_%"] = result["top5_perfect"] / total * 100
    else:
        result["top1_rate_%"] = 0
        result["top3_rate_%"] = 0
        result["top5_rate_%"] = 0

    return result


def main():
    """主函数 - 遍历 test 目录中的所有 JSON 文件进行评估"""
    # 检查测试目录
    if not os.path.exists(TEST_DIR):
        print(f"Error: Test directory not found: {TEST_DIR}")
        return

    # 检查模型文件
    if not os.path.exists(SAVED_MODEL_PATH):
        print(f"Error: Model checkpoint not found: {SAVED_MODEL_PATH}")
        print("Please ensure mergebert.pt is in saved_models/")
        return

    # 获取所有 JSON 文件
    json_files = glob.glob(os.path.join(TEST_DIR, "*.json"))
    # 排除已有的 eval 结果文件
    json_files = [f for f in json_files if not f.endswith("eval.json")]

    if not json_files:
        print(f"No JSON files found in {TEST_DIR}")
        return

    print(f"Found {len(json_files)} JSON files to evaluate:")
    for f in json_files:
        print(f"  - {os.path.basename(f)}")
    print()

    # 预加载模型（只加载一次）
    print("Loading model...")
    model, tokenizer, device = load_model_for_eval()
    print()

    # 汇总所有文件的结果
    all_metrics = {}

    # 遍历每个 JSON 文件
    for input_file in json_files:
        file_name = os.path.basename(input_file)
        base_name = os.path.splitext(file_name)[0]
        output_file = os.path.join(TEST_DIR, f"{base_name}eval.json")

        print(f"\n{'='*60}")
        print(f"Evaluating: {file_name}")
        print(f"{'='*60}")

        # 加载数据
        try:
            with open(input_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
            print(f"Loaded {len(data)} samples")
        except Exception as e:
            print(f"Error loading {file_name}: {e}")
            continue

        # 处理每个样本
        results = []
        for item in tqdm(data, desc=f"Processing {base_name}"):
            # 提取输入
            a_text = item.get("a_contents", "")
            b_text = item.get("b_contents", "")
            o_text = item.get("o_contents", "")

            # 获取 top-k 预测
            answers = get_topk_predictions(model, tokenizer, device, a_text, b_text, o_text, k=TOP_K)

            # 创建结果对象（保留原始字段并添加 answers）
            result_item = item.copy()
            result_item["answers"] = answers
            results.append(result_item)

        # 保存结果
        print(f"Saving results to: {output_file}")
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(results, f, ensure_ascii=False, indent=2)

        # 计算指标
        metrics = count_topk_perfect_match(results, "answers")
        all_metrics[file_name] = metrics

        print(f"\nResults for {file_name}:")
        print(f"  Total samples: {metrics['total']}")
        print(f"  Top-1 Perfect Match: {metrics['top1_perfect']} ({metrics['top1_rate_%']:.2f}%)")
        print(f"  Top-3 Perfect Match: {metrics['top3_perfect']} ({metrics['top3_rate_%']:.2f}%)")
        print(f"  Top-5 Perfect Match: {metrics['top5_perfect']} ({metrics['top5_rate_%']:.2f}%)")

    # 打印汇总结果
    print(f"\n{'='*60}")
    print("SUMMARY - All Files")
    print(f"{'='*60}")

    total_samples = 0
    total_top1 = 0
    total_top3 = 0
    total_top5 = 0

    for file_name, metrics in all_metrics.items():
        total_samples += metrics['total']
        total_top1 += metrics['top1_perfect']
        total_top3 += metrics['top3_perfect']
        total_top5 += metrics['top5_perfect']

    print(f"\n{'File':<40} {'Samples':<10} {'Top1%':<10} {'Top3%':<10} {'Top5%':<10}")
    print("-" * 80)
    for file_name, metrics in all_metrics.items():
        print(f"{file_name:<40} {metrics['total']:<10} {metrics['top1_rate_%']:<10.2f} {metrics['top3_rate_%']:<10.2f} {metrics['top5_rate_%']:<10.2f}")
    print("-" * 80)

    if total_samples > 0:
        overall_top1 = total_top1 / total_samples * 100
        overall_top3 = total_top3 / total_samples * 100
        overall_top5 = total_top5 / total_samples * 100
        print(f"{'OVERALL':<40} {total_samples:<10} {overall_top1:<10.2f} {overall_top3:<10.2f} {overall_top5:<10.2f}")

    print(f"{'='*60}")

    return all_metrics


if __name__ == "__main__":
    main()
